Prepared By **TAUFIQ MUSTAFIZUR RAHMAN**


---





**Task:**

### **Problem Statement: Alpha-Beta Pruning**

Consider a two-player, zero-sum game represented by the given game tree. The root node (Node A) represents the maximizing player's turn (MAX). The tree alternates between MAX and MIN layers at each subsequent depth. The terminal nodes at the bottom of the tree represent the final utility values of the game states.

Given the game tree with the terminal node values (from left to right): 2, 3, 5, 9, 0, 1, 7, and 5.

**Tasks:**

1. **Algorithm Tracing:** Traverse the tree from left to right using the Minimax algorithm combined with Alpha-Beta pruning.
2. **Node Evaluation:** Determine the final propagated utility values for all intermediate nodes (Nodes B, C, D, E, F, and G).
3. **Identify Pruning:** Identify exactly which terminal nodes or branches are pruned during the search. State the specific condition ($\alpha \ge \beta$) that caused each pruning action to occur.
4. **Optimal Strategy:** What is the final optimal utility value for the root node A, and what is the optimal path of play?

In [1]:
def alpha_beta(node_idx, depth, is_max, alpha, beta, leaf_values):
    """
    Evaluates the game tree using Minimax with Alpha-Beta Pruning.
    """
    # Base Case: We've reached the terminal nodes (depth 3)
    if depth == 3:
        return leaf_values[node_idx]

    # Maximizing Player's Turn (Nodes A, D, E, F, G)
    if is_max:
        best_val = -float('inf')
        # Iterate through the two binary children
        for i in range(2):
            child_idx = node_idx * 2 + i
            val = alpha_beta(child_idx, depth + 1, False, alpha, beta, leaf_values)
            best_val = max(best_val, val)
            alpha = max(alpha, best_val)

            # Pruning condition
            if beta <= alpha:
                print(f"-> Pruning occurred at MAX node {node_idx} at depth {depth}. (beta={beta} <= alpha={alpha})")
                break
        return best_val

    # Minimizing Player's Turn (Nodes B, C)
    else:
        best_val = float('inf')
        # Iterate through the two binary children
        for i in range(2):
            child_idx = node_idx * 2 + i
            val = alpha_beta(child_idx, depth + 1, True, alpha, beta, leaf_values)
            best_val = min(best_val, val)
            beta = min(beta, best_val)

            # Pruning condition
            if beta <= alpha:
                print(f"-> Pruning occurred at MIN node {node_idx} at depth {depth}. (beta={beta} <= alpha={alpha})")
                break
        return best_val

leaf_nodes = [2, 3, 5, 9, 0, 1, 7, 5]
optimal_value = alpha_beta(0,0,True,-float('inf'),float('inf'),leaf_nodes)
print(f"\nFinal Optimal Utility Value for the root: {optimal_value}")

-> Pruning occurred at MAX node 1 at depth 2. (beta=3 <= alpha=5)
-> Pruning occurred at MIN node 1 at depth 1. (beta=1 <= alpha=3)

Final Optimal Utility Value for the root: 3


**Task:**

[Problem Statement](https://docs.google.com/document/d/1M8GNijaRmbrEgLEJ8Sgi__-DYyj20kpJ/edit)

In bioinformatics and AI, generating a gene sequence that closely matches a desired target sequence (representing the optimal or healthy gene) is a key problem.
In this task, two AI agents take turns selecting nucleotides (gene elements) from a shared pool. Goal of this game is to build a gene sequence that matches a target gene sequence as closely as possible.


In [ ]:
def calculate_utility(gene, target, sid, booster_idx=-1):
    """Calculates the utility score based on ASCII differences and weights."""
    n = max(len(gene), len(target)) # N = Max length of the gene sequence or target
    target_len = len(target)

    # Weights for each position correspond to the last n digits of your student ID
    sid_str = str(sid)
    base_weights = [int(x) for x in sid_str[-target_len:]]
    booster_factor = int(sid_str[:2]) / 100.0

    utility = 0
    for i in range(n):
        if i < target_len:
            w = base_weights[i]
        else:
            w = 1  # Weight at position i if available, otherwise 1

        # Apply genetic booster if activated at or before this index
        if booster_idx != -1 and i >= booster_idx:
            w = w * booster_factor

        # Calculate ASCII distance
        char_gene = ord(gene[i]) if i < len(gene) else 0
        char_target = ord(target[i]) if i < len(target) else 0

        utility -= w * abs(char_gene - char_target)

    return utility

def minimax(pool, target, sid, is_max, alpha, beta, current_gene, booster_idx):
    """Recursive Minimax with Alpha-Beta Pruning."""
    # Terminal State
    if len(pool) == 0:
        return calculate_utility(current_gene, target, sid, booster_idx), current_gene

    if is_max:
        max_eval = -float('inf')
        best_gene = ""
        for i, char in enumerate(pool):
            new_pool = pool[:i] + pool[i+1:]
            new_gene = current_gene + char

            # If Agent 1 (Maximizer) picks 'S', it activates a genetic booster
            new_booster = booster_idx
            if char == 'S' and booster_idx == -1:
                new_booster = len(current_gene)

            eval_score, sequence = minimax(new_pool, target, sid, False, alpha, beta, new_gene, new_booster)

            if eval_score > max_eval:
                max_eval = eval_score
                best_gene = sequence

            alpha = max(alpha, eval_score)
            if beta <= alpha:
                break
        return max_eval, best_gene

    else:
        min_eval = float('inf')
        best_gene = ""
        for i, char in enumerate(pool):
            new_pool = pool[:i] + pool[i+1:]
            new_gene = current_gene + char

            # Agent 2 plays normally, no booster activation even if 'S' is encountered[cite: 2]
            eval_score, sequence = minimax(new_pool, target, sid, True, alpha, beta, new_gene, booster_idx)

            if eval_score < min_eval:
                min_eval = eval_score
                best_gene = sequence

            beta = min(beta, eval_score)
            if beta <= alpha:
                break
        return min_eval, best_gene

def solve_gene_sequence(pool_input, target, sid="24301485"):
    """Runs both Task I and Task II and compares the outcomes."""
    pool = pool_input.split(',')

    # --- Task I Execution ---
    score_t1, gene_t1 = minimax(pool, target, sid, True, -float('inf'), float('inf'), "", -1)

    # --- Task II Execution ---
    # The pool may include a special nucleotide 'S' at the rightmost position[cite: 2]
    pool_t2 = pool + ['S']
    score_t2, gene_t2 = minimax(pool_t2, target, sid, True, -float('inf'), float('inf'), "", -1)


    print(f"Sample Input:\n{pool_input}\n{target}\n{sid}\n")
    print(f"--- Task I Output ---")
    print(f"Best gene sequence generated: {gene_t1}")
    print(f"Utility score: {int(score_t1) if score_t1.is_integer() else score_t1}")

    print(f"\n--- Task II Output ---")
    if score_t2 > score_t1:
        print("YES\nWith special nucleotide")
    else:
        print("NO\nWithout special nucleotide")

    print(f"Best gene sequence generated: {gene_t2}")

    # Formatting output appropriately based on whether the result is a float
    formatted_score = f"{score_t2:.2f}" if not float(score_t2).is_integer() else f"{int(score_t2)}"
    print(f"Utility score: {formatted_score}")


if __name__ == "__main__":
    solve_gene_sequence("A,T,C,G", "ATGC")